In [1]:
import os
import time
import sys
from pathlib import Path

import json
import uuid

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql.functions import coalesce, col, lit
from pyspark.sql.types import (
    LongType,
    StringType,
    StructField,
    StructType,
)

from etl.transformations.common import (
    create_spark,
    read_clickhouse,
    read_current_snapshot,
    write_clickhouse,
    list_batches
)

from etl.transformations.dimensions.scd.common import (
    build_expired_version,
    build_new_version,
    add_hash,
)

MINIO_STAGING_BUCKET = os.environ.get(
    "MINIO_STAGING_BUCKET",
    "staging",
)

In [2]:
def transform_dim_user_farm_role(
    user_role_df: DataFrame,
    user_df: DataFrame,
    role_df: DataFrame,
    dim_farm_df: DataFrame,
) -> DataFrame:
    """
    Build the dim_user_farm_role source dataset.

    Uses dim_farm surrogate key because dim_farm is SCD2.
    """

    return (
        user_role_df.join(
            user_df,
            user_role_df.user_id == user_df.id,
            "left",
        )
        .join(
            role_df,
            user_role_df.role_id == role_df.id,
            "left",
        )
        .join(
            dim_farm_df,
            user_role_df.farm_id == dim_farm_df.farm_id,
            "left",
        )
        .select(
            user_role_df.id.alias("user_role_id"),
            user_role_df.user_id,
            user_role_df.role_id,
            coalesce(
                dim_farm_df.farm_key,
                lit(0),
            ).alias("farm_key"),
            user_role_df.farm_id,
            user_df.full_name.alias("user_full_name"),
            role_df.name.alias("role_name"),
            dim_farm_df.name.alias("farm_name"),
        )
    )

In [3]:
def main():
    """
    Load dim_user_farm_role as an SCD2 dimension.
    """

    spark = create_spark("load_dim_user_farm_role")

    try:
        USER_ROLE_SCHEMA = StructType(
            [
                StructField("id", LongType(), True),
                StructField("user_id", LongType(), True),
                StructField("role_id", LongType(), True),
                StructField("farm_id", LongType(), True),
                StructField("created_at", LongType(), True),
                StructField("updated_at", LongType(), True),
            ]
        )

        # Reconstruct current PostgreSQL state from all MinIO batches.
        user_role_df = read_current_snapshot(
            spark,
            MINIO_STAGING_BUCKET,
            "user_roles",
            schema=USER_ROLE_SCHEMA,
        )

        user_role_df.show()

        user_df = read_current_snapshot(
            spark,
            MINIO_STAGING_BUCKET,
            "users",
        )

        role_df = read_current_snapshot(
            spark,
            MINIO_STAGING_BUCKET,
            "roles",
        )

        dim_farm_df = read_clickhouse(
            spark,
            """
                (
                    SELECT *
                    FROM dim_farm FINAL
                    WHERE is_current = 1
                ) AS dim_farm
            """,
        )

        # Build the complete current dimension state.
        dim_user_farm_role_source_df = transform_dim_user_farm_role(
            user_role_df,
            user_df,
            role_df,
            dim_farm_df,
        )

        # Read current active SCD2 versions.
        current_dim_df = read_clickhouse(
            spark,
            """
                (
                    SELECT *
                    FROM dim_user_farm_role FINAL
                ) AS dim_user_farm_role
            """,
        ).filter(
            col("is_current") == 1,
        )

        hash_columns = [
            "user_id",
            "role_id",
            "farm_key",
            "farm_id",
            "user_full_name",
            "role_name",
            "farm_name",
        ]

        source_hashed_df = add_hash(
            dim_user_farm_role_source_df,
            hash_columns,
        )

        current_hashed_df = add_hash(
            current_dim_df,
            hash_columns,
        )

        # Compare source state with active warehouse state.
        comparison_df = source_hashed_df.alias("source").join(
            current_hashed_df.alias(
                "current",
            ),
            "user_role_id",
            "left",
        )

        # New relationships.
        new_rows_df = comparison_df.filter(col("current.user_role_id").isNull()).select(
            "source.*",
        )

        # Existing relationships where attributes changed.
        changed_rows_df = comparison_df.filter(
            (col("current.user_role_id").isNotNull())
            & (col("source._hash") != col("current._hash"))
        )

        # Old versions to close.
        expired_rows_df = changed_rows_df.select(
            "current.*",
        )

        # New active versions.
        new_versions_df = changed_rows_df.select(
            "source.*",
        )

        load_version = int(
            time.time() * 1000,
        )

        rows_to_write = (
            build_new_version(
                new_rows_df,
                load_version,
                [
                    "user_role_key",
                    "farm_key",
                ],
            )
            .unionByName(
                build_expired_version(
                    expired_rows_df,
                    load_version,
                    [
                        "user_role_key",
                        "farm_key",
                    ],
                )
            )
            .unionByName(
                build_new_version(
                    new_versions_df,
                    load_version,
                    [
                        "user_role_key",
                        "farm_key",
                    ],
                )
            )
        )

        if rows_to_write.count() > 0:
            write_clickhouse(
                rows_to_write,
                "dim_user_farm_role",
            )

    finally:
        spark.stop()

In [4]:
if __name__ == "__main__":
    main()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/22 14:10:59 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/07/22 14:11:02 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


+---+-------+-------+-------+----------+----------+
| id|user_id|role_id|farm_id|created_at|updated_at|
+---+-------+-------+-------+----------+----------+
|  1|      1|      3|   NULL|1784710849|1784710849|
|  2|      2|      1|      1|1784723689|1784723689|
+---+-------+-------+-------+----------+----------+



26/07/22 14:11:13 WARN JdbcUtils: Requested isolation level 1, but transactions are unsupported
26/07/22 14:11:13 WARN JdbcUtils: Requested isolation level 1, but transactions are unsupported
26/07/22 14:11:13 WARN JdbcUtils: Requested isolation level 1, but transactions are unsupported
